# M6 — Marquette 12SL reference model (combined)

Single self-contained notebook for the **M6 reference model**, merging the former 03 (features), 04 (A/B/C/D selection) and 05 (modeling) — nothing removed, rigor added (standardized evaluation, IC95, multi-seed, gap, re-run guards, non-regression check).

**Role:** external gold-standard reference (Marquette 12SL, PTB-XL only, 57 WPW). NOT in the deployed ensemble; NEVER touches fold 10.

**Inputs:** `data/raw/ptbxl_plus/12sl_features.csv`, `data/processed/metadata_ptbxl.csv`.
**Outputs:** `models/M6_marquette/` (config + models), `reports/{figures,metrics}/`.

> Legacy notebooks 03/04/05 are kept as reference; this notebook supersedes them (final renumbering later). Narrative/journal: see `JOURNAL.md`.

### Block M6.0: Setup

In [ ]:
# M6 = Marquette 12SL reference model. PTB-XL ONLY (Marquette features do not exist for Ningbo),
# 57 WPW in training folds. Role: external gold-standard reference for the paper, NOT part of the
# deployed ensemble, and it NEVER touches fold 10. No signal filter here (features are precomputed).
import os, json, sys
import numpy as np, pandas as pd

ROOT       = r'.'
PROCESSED  = os.path.join(ROOT, 'data', 'processed')
PTBXL_PLUS = os.path.join(ROOT, 'data', 'raw', 'ptbxl_plus')
MODELS     = os.path.join(ROOT, 'models', 'M6_marquette')
FIG        = os.path.join(ROOT, 'reports', 'figures')
METRICS    = os.path.join(ROOT, 'reports', 'metrics')
for d in (MODELS, FIG, METRICS): os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(ROOT, 'src'))

FORCE_REBUILD = False   # re-run guard for the feature table (Block M6.1)

print("=== Setup ===")
for p in [PROCESSED, PTBXL_PLUS, os.path.join(PROCESSED, 'metadata_ptbxl.csv')]:
    print(f"  {'OK     ' if os.path.exists(p) else 'MISSING'} {os.path.relpath(p, ROOT)}")

### Block M6.1: Marquette feature table

In [ ]:
# Build features_marquette.csv: join PTB-XL metadata (label/fold) with the 782 Marquette 12SL
# features from PTB-XL+. Skipped if the file already exists.
import os, pandas as pd
FEAT_CSV = os.path.join(PROCESSED, 'features_marquette.csv')

if os.path.exists(FEAT_CSV) and not FORCE_REBUILD:
    fm = pd.read_csv(FEAT_CSV)
    print(f"features_marquette.csv exists -> SKIPPED build (loaded {fm.shape[0]}x{fm.shape[1]}).")
else:
    df12 = pd.read_csv(os.path.join(PTBXL_PLUS, '12sl_features.csv'))
    meta = pd.read_csv(os.path.join(PROCESSED, 'metadata_ptbxl.csv'))
    fm = meta[['ecg_id','patient_id','label','fold','source']].merge(df12, on='ecg_id', how='inner')
    fm.to_csv(FEAT_CSV, index=False)
    print(f"Built features_marquette.csv: {fm.shape[0]} rows x {fm.shape[1]} cols")

n_feat = fm.shape[1] - 5
print(f"PTB-XL: {len(fm)} ECGs | WPW {int((fm.label==1).sum())} | {n_feat} Marquette features")
assert len(fm) == 21799 and int((fm.label==1).sum()) == 70, "unexpected PTB-XL feature table"

### Block M6.2a: Selection A (clinical)

In [ ]:
# Selection A (clinical): Cohen's d on the 22 Global clinical features (folds 1-8).
# Gate |d|>0.3 AND p<0.05; exclude artifacts (QRS_On/Q_On: absolute position, bimodal);
# de-duplicate the QT family (keep Framingham); cap Y=6 (events-per-variable on 57 WPW).
import os, json, pandas as pd, numpy as np
from scipy import stats

fm = pd.read_csv(os.path.join(PROCESSED, 'features_marquette.csv'))
train = fm[fm['fold'].between(1, 8)].copy()

global_clinical = ['PR_Int_Global','QRS_Dur_Global','QRS_On_Global','QRS_Off_Global',
    'P_On_Global','P_Off_Global','P_Dur_Global','Q_On_Global','Q_Off_Global','T_On_Global','T_Off_Global',
    'QT_Int_Global','QT_IntBazett_Global','QT_IntCorr_Global','QT_IntFridericia_Global','QT_IntFramingham_Global',
    'P_AxisFront_Global','R_AxisFrontal_Global','T_AxisFront_Global','HR_Ventr_Global','RR_Mean_Global','HR_Atrial_Global']
global_clinical = [c for c in global_clinical if c in train.columns]

def cohen_d(a, b):
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return np.nan
    ps = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    return (a.mean() - b.mean()) / ps if ps > 0 else np.nan

wpw = train['label'] == 1
rows = []
for col in global_clinical:
    w, n = train.loc[wpw, col].values, train.loc[~wpw, col].values
    d = cohen_d(w, n); a, b = w[~np.isnan(w)], n[~np.isnan(n)]
    _, p = stats.mannwhitneyu(a, b, alternative='two-sided') if len(a) >= 2 and len(b) >= 2 else (np.nan, np.nan)
    rows.append({'feature': col, 'cohen_d': d, 'abs_d': abs(d) if not np.isnan(d) else np.nan, 'p_value': p})
res_A = pd.DataFrame(rows).sort_values('abs_d', ascending=False).reset_index(drop=True)
res_A['pass_gate'] = (res_A['abs_d'] > 0.3) & (res_A['p_value'] < 0.05)

ARTEFACTS = ['QRS_On_Global', 'Q_On_Global']
QT_REDUNDANT = ['QT_IntFridericia_Global', 'QT_Int_Global', 'QT_IntCorr_Global', 'QT_IntBazett_Global']
cand = res_A[res_A['pass_gate'] & ~res_A['feature'].isin(ARTEFACTS + QT_REDUNDANT)]
Y = 6
selected_A = cand.sort_values('abs_d', ascending=False).head(Y)['feature'].tolist()

print(f"A candidates passing gate: {int(res_A['pass_gate'].sum())} | after artifact/QT removal -> top {Y}")
for f in selected_A:
    print(f"  {f:26s} |d|={res_A.set_index('feature').loc[f,'abs_d']:.3f}")
json.dump({'model':'M6_A_clinical','n_features':len(selected_A),'features':selected_A,'Y_max':Y,
           'criteria':"|d|>0.3 AND p<0.05; QRS_On/Q_On excluded as artifacts; QT de-duplicated (Framingham); cap 6"},
          open(os.path.join(PROCESSED,'selection_A.json'),'w',encoding='utf-8'), ensure_ascii=False, indent=2)

### Block M6.2b: Selection B (discovery families)

In [ ]:
# Selection B (discovery): scan all 782 per-lead Marquette features (folds 1-8) with Benjamini-Hochberg
# FDR, aggregate each family by mean over the 12 leads, re-test, de-duplicate (corr>0.9), cap Y'=10.
# MAJOR finding: per-lead R_PeakTime / R_Dur families beat the clinical PR interval.
import os, json, re, pandas as pd, numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

fm = pd.read_csv(os.path.join(PROCESSED, 'features_marquette.csv'))
meta_cols = ['ecg_id','patient_id','label','fold','source']
leads = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']

def cohen_d(a, b):
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return np.nan
    ps = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    return (a.mean() - b.mean()) / ps if ps > 0 else np.nan

# --- 1) raw per-feature ranking on the 782 features (BH FDR), saved for the record ---
train = fm[fm['fold'].between(1, 8)].copy()
feat_cols = [c for c in train.columns if c not in meta_cols]
wpw = train['label'] == 1
rows = []
for col in feat_cols:
    w, n = train.loc[wpw, col].values, train.loc[~wpw, col].values
    d = cohen_d(w, n); a, b = w[~np.isnan(w)], n[~np.isnan(n)]
    try: _, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    except ValueError: p = np.nan
    rows.append({'feature': col, 'cohen_d': d, 'abs_d': abs(d) if not np.isnan(d) else np.nan, 'p_value': p})
res_B = pd.DataFrame(rows)
ok = res_B['p_value'].notna()
res_B.loc[ok, 'p_bh'] = multipletests(res_B.loc[ok, 'p_value'], alpha=0.05, method='fdr_bh')[1]
res_B = res_B.sort_values('abs_d', ascending=False).reset_index(drop=True)
res_B.to_csv(os.path.join(PROCESSED, 'discovery_ranking.csv'), index=False)

# --- 2) aggregate per-lead families by MEAN over leads ---
def family_of(col):
    for l in sorted(leads, key=len, reverse=True):
        if col.endswith('_' + l): return col[:-(len(l)+1)]
    return None
families = {}
for col in fm.columns:
    if col in meta_cols: continue
    fam = family_of(col)
    if fam is not None: families.setdefault(fam, []).append(col)
families = {f: cols for f, cols in families.items() if len(cols) >= 6}
agg = pd.DataFrame({'ecg_id': fm['ecg_id'], 'label': fm['label'], 'fold': fm['fold']})
for fam, cols in families.items():
    agg[fam + '__mean'] = fm[cols].mean(axis=1)
agg.to_csv(os.path.join(PROCESSED, 'discovery_families_mean.csv'), index=False)
print(f"{len(feat_cols)} per-lead features -> {len(families)} families aggregated by mean")

# --- 3) gate families (|d|>0.3 AND p_BH<0.05), de-duplicate corr>0.9, cap Y'=10 ---
tr = agg[agg['fold'].between(1, 8)].copy()
fam_cols = [c for c in agg.columns if c.endswith('__mean')]
wpw = tr['label'] == 1
rows = []
for col in fam_cols:
    w, n = tr.loc[wpw, col].values, tr.loc[~wpw, col].values
    d = cohen_d(w, n); a, b = w[~np.isnan(w)], n[~np.isnan(n)]
    _, p = stats.mannwhitneyu(a, b, alternative='two-sided') if len(a) >= 2 and len(b) >= 2 else (np.nan, np.nan)
    rows.append({'family': col, 'abs_d': abs(d) if not np.isnan(d) else np.nan, 'p_value': p})
dfam = pd.DataFrame(rows)
dfam['p_bh'] = multipletests(dfam['p_value'].fillna(1), alpha=0.05, method='fdr_bh')[1]
dfam['pass_gate'] = (dfam['abs_d'] > 0.3) & (dfam['p_bh'] < 0.05)
cand = dfam[dfam['pass_gate']].sort_values('abs_d', ascending=False).reset_index(drop=True)

corr = tr[cand['family'].tolist()].corr().abs()
kept = []
for f in cand['family']:
    if all(corr.loc[f, k] <= 0.9 for k in kept): kept.append(f)
Yp = 10
selected_B = kept[:Yp]
print(f"B families passing gate: {int(dfam['pass_gate'].sum())} | after dedup -> top {Yp}")
for f in selected_B:
    print(f"  {f:24s} |d|={cand.set_index('family').loc[f,'abs_d']:.3f}")
json.dump({'model':'M6_B_discovery','n_features':len(selected_B),'features':selected_B,'Yp_max':Yp,
           'criteria':"|d|>0.3 AND BH<0.05; families aggregated by mean; dedup corr>0.9; cap 10"},
          open(os.path.join(PROCESSED,'selection_B.json'),'w',encoding='utf-8'), ensure_ascii=False, indent=2)

### Block M6.2c: Selections C (combined) and D (control)

In [ ]:
# Selection C (combined = all of A + best of B up to 10) and D (control "scarecrow": every gated
# family, NO cap -> deliberately violates events-per-variable to demonstrate overfitting in M6.3).
import os, json, pandas as pd, numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

A = json.load(open(os.path.join(PROCESSED, 'selection_A.json'), encoding='utf-8'))['features']
B = json.load(open(os.path.join(PROCESSED, 'selection_B.json'), encoding='utf-8'))['features']

# --- Liste C : A (all) + best of B until budget 10 ---
Ypp = 10
selected_C = list(A)
for feat in B:
    if len(selected_C) >= Ypp: break
    if feat not in selected_C: selected_C.append(feat)
print(f"Selection C (combined): {len(A)} from A + {len(selected_C)-len(A)} from B")
for f in selected_C: print(f"  [{'A' if f in A else 'B'}] {f}")
json.dump({'model':'M6_C_combined','n_features':len(selected_C),'features':selected_C,'Ypp_max':Ypp},
          open(os.path.join(PROCESSED,'selection_C.json'),'w',encoding='utf-8'), ensure_ascii=False, indent=2)

# --- Liste D : control, all gated families, no cap ---
agg = pd.read_csv(os.path.join(PROCESSED, 'discovery_families_mean.csv'))
tr = agg[agg['fold'].between(1, 8)].copy()
fam_cols = [c for c in agg.columns if c.endswith('__mean')]
def cohen_d(a, b):
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return np.nan
    ps = np.sqrt((a.std(ddof=1)**2 + b.std(ddof=1)**2) / 2)
    return (a.mean() - b.mean()) / ps if ps > 0 else np.nan
wpw = tr['label'] == 1
rows = []
for col in fam_cols:
    w, n = tr.loc[wpw, col].values, tr.loc[~wpw, col].values
    d = cohen_d(w, n); a, b = w[~np.isnan(w)], n[~np.isnan(n)]
    _, p = stats.mannwhitneyu(a, b, alternative='two-sided') if len(a) >= 2 and len(b) >= 2 else (np.nan, np.nan)
    rows.append({'family': col, 'abs_d': abs(d) if not np.isnan(d) else np.nan, 'p_value': p})
dfam = pd.DataFrame(rows)
dfam['p_bh'] = multipletests(dfam['p_value'].fillna(1), alpha=0.05, method='fdr_bh')[1]
cand = dfam[(dfam['abs_d'] > 0.3) & (dfam['p_bh'] < 0.05)].sort_values('abs_d', ascending=False).reset_index(drop=True)
corr = tr[cand['family'].tolist()].corr().abs()
selected_D = []
for f in cand['family']:
    if all(corr.loc[f, k] <= 0.9 for k in selected_D): selected_D.append(f)
n_wpw_tr = int((tr['label'] == 1).sum())
print(f"\nSelection D (control, no cap): {len(selected_D)} families | EPV ratio {n_wpw_tr/len(selected_D):.2f} WPW/feature (<< 10 -> overfit expected)")
json.dump({'model':'M6_D_control_noCAP','n_features':len(selected_D),'features':selected_D,'cap':'NONE',
           'role':'control to demonstrate overfitting vs capped A/B/C',
           'epv_wpw_per_feature':round(n_wpw_tr/len(selected_D),2)},
          open(os.path.join(PROCESSED,'selection_D.json'),'w',encoding='utf-8'), ensure_ascii=False, indent=2)

### Block M6.2d: Selection histograms

In [ ]:
# Selection histograms (validation visuals kept from 04): top-6 of list A (clinical) and of the
# discovery families (B), WPW vs non-WPW on folds 1-8.
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
fm  = pd.read_csv(os.path.join(PROCESSED, 'features_marquette.csv'))
agg = pd.read_csv(os.path.join(PROCESSED, 'discovery_families_mean.csv'))
A = json.load(open(os.path.join(PROCESSED, 'selection_A.json'), encoding='utf-8'))['features']
B = json.load(open(os.path.join(PROCESSED, 'selection_B.json'), encoding='utf-8'))['features']

def hist_panel(src, feats, title, fname, color):
    tr = src[src.fold.between(1, 8)]
    fig, ax = plt.subplots(2, 3, figsize=(15, 7))
    for j, f in enumerate(feats[:6]):
        a = ax[j//3, j%3]
        w = tr.loc[tr.label==1, f].dropna(); n = tr.loc[tr.label==0, f].dropna()
        lo, hi = np.nanpercentile(tr[f], 1), np.nanpercentile(tr[f], 99)
        bins = np.linspace(lo, hi, 40)
        a.hist(n, bins=bins, density=True, alpha=.5, color='#94a3b8', label='non-WPW')
        a.hist(w, bins=bins, density=True, alpha=.6, color=color, label='WPW')
        a.set_title(f, fontsize=9); a.legend(fontsize=7)
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=130, bbox_inches='tight'); plt.show()

hist_panel(fm,  A, 'M6 list A (clinical) — top 6 features (folds 1-8)', 'A_global_histograms.png', '#d62728')
hist_panel(agg, B, 'M6 list B (discovery families) — top 6 (folds 1-8)', 'B_discovery_histograms.png', '#2ca02c')
print("Saved A_global_histograms.png, B_discovery_histograms.png")

### Block M6.3: Multi-algo tournament (A/B/C/D x LR/RF/XGB)

In [ ]:
# Multi-algo tournament (LogReg / RandomForest / XGBoost) x lists A/B/C/D, CV on folds 1-8.
# PRESERVED from the original analysis. Added rigor: AP (project's primary judge) and the
# train-minus-OOF AP gap, which quantifies overfitting (list D = highest gap = the scarecrow).
import os, json, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

fm  = pd.read_csv(os.path.join(PROCESSED, 'features_marquette.csv'))
agg = pd.read_csv(os.path.join(PROCESSED, 'discovery_families_mean.csv'))
data = fm.merge(agg.drop(columns=['label','fold']), on='ecg_id')
sels = {k: json.load(open(os.path.join(PROCESSED, f'selection_{k}.json'), encoding='utf-8'))['features']
        for k in ['A','B','C','D']}

d8 = data[data.fold.between(1,8)].reset_index(drop=True)
y, folds = d8.label.values, d8.fold.values
spw = (y==0).sum() / (y==1).sum()

def make_lr():  return Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler()),
                                 ('lr',LogisticRegression(class_weight='balanced', max_iter=1000))])
def make_rf():  return Pipeline([('imp',SimpleImputer(strategy='median')),
                                 ('rf',RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=5,
                                       max_features='sqrt', class_weight='balanced', random_state=42, n_jobs=-1))])
def make_xgb(): return XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8,
                                     colsample_bytree=0.8, reg_lambda=2.0, min_child_weight=3, scale_pos_weight=spw,
                                     eval_metric='auc', tree_method='hist', random_state=42, n_jobs=-1)
MAKERS = {'LogReg': make_lr, 'RandomForest': make_rf, 'XGBoost': make_xgb}

def evaluate(feats, make):
    oof = np.full(len(d8), np.nan); fold_aucs = []
    for h in range(1, 9):
        trm, vam = folds != h, folds == h
        if y[vam].sum() == 0: continue
        m = make(); m.fit(d8.loc[trm, feats], y[trm]); p = m.predict_proba(d8.loc[vam, feats])[:,1]
        oof[vam] = p; fold_aucs.append(roc_auc_score(y[vam], p))
    mk = ~np.isnan(oof)
    ap_oof = average_precision_score(y[mk], oof[mk]); auc_oof = roc_auc_score(y[mk], oof[mk])
    full = make(); full.fit(d8[feats], y); ap_tr = average_precision_score(y, full.predict_proba(d8[feats])[:,1])
    return np.mean(fold_aucs), auc_oof, ap_oof, ap_tr - ap_oof

rows = []
for lname, feats in sels.items():
    for aname, make in MAKERS.items():
        auc_cv, auc_oof, ap_oof, gap = evaluate(feats, make)
        rows.append({'list': lname, 'algo': aname, 'n_feat': len(feats),
                     'AUC_cv_mean': auc_cv, 'AUC_oof': auc_oof, 'AP_oof': ap_oof, 'gap_AP': gap})
res = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(res.sort_values(['list','algo']).to_string(index=False))
print("\nRead: C + XGBoost is best; list D shows the highest gap_AP (overfitting) across all 3 algos -> the scarecrow works.")

### Block M6.4: Hyperparameter grid (list C)

In [ ]:
# Disciplined hyperparameter grid for XGBoost on list C (CV folds 1-8). Justifies depth=2:
# best nominal AUC is within CV noise of depth=2, and depth=2 is the most stable (lowest std).
import os, json, numpy as np, pandas as pd
from itertools import product
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

fm  = pd.read_csv(os.path.join(PROCESSED, 'features_marquette.csv'))
agg = pd.read_csv(os.path.join(PROCESSED, 'discovery_families_mean.csv'))
data = fm.merge(agg.drop(columns=['label','fold']), on='ecg_id')
feats = json.load(open(os.path.join(PROCESSED, 'selection_C.json'), encoding='utf-8'))['features']
d8 = data[data.fold.between(1,8)].reset_index(drop=True); y, folds = d8.label.values, d8.fold.values
spw = (y==0).sum()/(y==1).sum()
fixed = dict(subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, min_child_weight=3,
             scale_pos_weight=spw, eval_metric='auc', tree_method='hist', random_state=42, n_jobs=-1)

rows = []
for md, lr, ne in product([2,3,4], [0.03,0.05], [100,200]):
    aucs = []
    for h in range(1, 9):
        trm, vam = folds != h, folds == h
        if y[vam].sum() == 0: continue
        m = XGBClassifier(max_depth=md, learning_rate=lr, n_estimators=ne, **fixed)
        m.fit(d8.loc[trm, feats], y[trm]); aucs.append(roc_auc_score(y[vam], m.predict_proba(d8.loc[vam, feats])[:,1]))
    rows.append({'max_depth':md,'learning_rate':lr,'n_estimators':ne,'AUC_cv_mean':np.mean(aucs),'AUC_cv_std':np.std(aucs)})
res = pd.DataFrame(rows).sort_values('AUC_cv_mean', ascending=False).reset_index(drop=True)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(res.to_string(index=False))
print("\nDecision: depth=2 retained for STABILITY (lowest std), gains at higher depth are within CV noise.")

### Block M6.5: Final M6 — OOF folds 1-8 (calibrated) + IC95 + multi-seed

In [ ]:
# Final M6 = list C + XGBoost depth=2, calibrated (Platt/sigmoid) OOF on NATIVE folds 1-8
# (anti-shuffle). Headline metrics use the 57 WPW of folds 1-8 (robust). Added rigor: IC95
# bootstrap on AP and AUC, plus multi-seed stability.
import os, json, numpy as np, pandas as pd
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

fm  = pd.read_csv(os.path.join(PROCESSED, 'features_marquette.csv'))
agg = pd.read_csv(os.path.join(PROCESSED, 'discovery_families_mean.csv'))
data = fm.merge(agg.drop(columns=['label','fold']), on='ecg_id')
feats = json.load(open(os.path.join(PROCESSED, 'selection_C.json'), encoding='utf-8'))['features']

train = data[data.fold.between(1,8)].reset_index(drop=True)
X, y = train[feats], train.label.values
spw = (y==0).sum()/(y==1).sum()

def make_xgb(seed=42):
    return XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, reg_lambda=2.0, min_child_weight=3, scale_pos_weight=spw,
        eval_metric='auc', tree_method='hist', random_state=seed, n_jobs=-1)

def oof_calibrated(seed=42):
    oof = np.zeros(len(train))
    for h in sorted(train.fold.unique()):
        tr = (train.fold != h).values; va = (train.fold == h).values
        cal = CalibratedClassifierCV(make_xgb(seed), method='sigmoid', cv=3)
        cal.fit(X[tr], y[tr]); oof[va] = cal.predict_proba(X[va])[:,1]
    return oof

oof = oof_calibrated(42)
AUC_oof = float(roc_auc_score(y, oof)); AP_oof = float(average_precision_score(y, oof))

# Operating threshold = F1-max on OOF (unconstrained); the frozen reference point.
g = pd.DataFrame([(t, f1_score(y,(oof>=t).astype(int),zero_division=0),
                      recall_score(y,(oof>=t).astype(int),zero_division=0),
                      precision_score(y,(oof>=t).astype(int),zero_division=0))
                  for t in np.linspace(0.01,0.99,99)], columns=['t','f1','recall','prec'])
best = g.loc[g.f1.idxmax()]; THR = float(round(best.t,3))
print(f"OOF folds 1-8 (calibrated): AUC {AUC_oof:.4f} | AP {AP_oof:.4f}")
print(f"F1-max threshold {THR:.3f} -> recall {best.recall:.3f} | precision {best.prec:.3f} | F1 {best.f1:.3f}")

# IC95 bootstrap (stratified pos/neg) on AUC and AP
rng = np.random.default_rng(42); pos = np.where(y==1)[0]; neg = np.where(y==0)[0]
ba = np.empty(2000); bp = np.empty(2000)
for b in range(2000):
    idx = np.concatenate([rng.choice(pos,len(pos),True), rng.choice(neg,len(neg),True)])
    ba[b] = roc_auc_score(y[idx], oof[idx]); bp[b] = average_precision_score(y[idx], oof[idx])
AUC_IC = [float(np.percentile(ba,2.5)), float(np.percentile(ba,97.5))]
AP_IC  = [float(np.percentile(bp,2.5)), float(np.percentile(bp,97.5))]
print(f"IC95  AUC [{AUC_IC[0]:.4f}, {AUC_IC[1]:.4f}] | AP [{AP_IC[0]:.4f}, {AP_IC[1]:.4f}]")

# Multi-seed stability (NEW): OOF AUC/AP over 10 seeds
msa = []; msp = []
for s in range(10):
    o = oof_calibrated(s); msa.append(roc_auc_score(y,o)); msp.append(average_precision_score(y,o))
MULTISEED = {'AUC_mean':float(np.mean(msa)),'AUC_std':float(np.std(msa)),
             'AP_mean':float(np.mean(msp)),'AP_std':float(np.std(msp))}
print(f"Multi-seed (10): AUC {MULTISEED['AUC_mean']:.4f}+/-{MULTISEED['AUC_std']:.4f} | "
      f"AP {MULTISEED['AP_mean']:.4f}+/-{MULTISEED['AP_std']:.4f}")

### Block M6.6: Standardized fold-9 evaluation + FN profile

In [ ]:
# Standardized evaluation on fold 9 (held-out, 6 WPW), via src/evaluation.evaluate_standard ->
# same metrics/figure/JSON as M1/M2 for an apples-to-apples comparison table. Threshold is F1-max
# on the OOF folds 1-8 (raw scores), applied to fold 9. Plus a short false-negative profile.
import os, numpy as np, pandas as pd
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from evaluation import evaluate_standard

f9 = data[data.fold == 9].reset_index(drop=True)
X9, y9 = f9[feats], f9.label.values

# Raw OOF (folds 1-8) for the threshold; raw + calibrated scores on fold 9.
oof_raw = np.zeros(len(train))
for h in sorted(train.fold.unique()):
    tr = (train.fold != h).values; va = (train.fold == h).values
    oof_raw[va] = make_xgb(42).fit(X[tr], y[tr]).predict_proba(X[va])[:,1]
raw9 = make_xgb(42).fit(X, y)
cal9 = CalibratedClassifierCV(make_xgb(42), method='sigmoid', cv=3).fit(X, y)
score9_raw = raw9.predict_proba(X9)[:,1]; score9_cal = cal9.predict_proba(X9)[:,1]
ap_train = float(average_precision_score(y, raw9.predict_proba(X)[:,1]))

M6_STD = evaluate_standard('M6_marquette', y_oof=y, score_oof=oof_raw, y_test=y9, score_test=score9_raw,
    figures_dir=FIG, metrics_dir=METRICS, score_test_calibrated=score9_cal, ap_train=ap_train,
    extra={'protocol':'threshold on OOF folds 1-8 (57 WPW), evaluated on fold 9 (6 WPW); reference model, PTB-XL only',
           'headline_oof_auc':AUC_oof, 'headline_oof_ap':AP_oof})
print(f"Standardized fold-9: AP {M6_STD['AP']:.3f} | AUC {M6_STD['AUC']:.3f} | "
      f"F1 {M6_STD['confusion_at_threshold']['f1']:.3f}")

# False-negative profile on OOF at the operating threshold (kept from the original analysis)
tr_fn = train.copy(); tr_fn['proba'] = oof; tr_fn['pred'] = (oof >= THR).astype(int)
wpw = tr_fn[tr_fn.label==1]; fn = wpw[wpw.pred==0]
print(f"OOF false negatives @ {THR:.3f}: {len(fn)}/{len(wpw)} missed | {(fn.proba<0.10).sum()} blinded (<0.10)")
clin = [c for c in ['QRS_Dur_Global','PR_Int_Global','R_PeakTime__mean'] if c in tr_fn.columns]
if clin:
    print("Median FN vs TP:", {c: (round(float(fn[c].median()),1), round(float(wpw[wpw.pred==1][c].median()),1)) for c in clin})
print("Suspect labels to inspect visually (kept): ecg_id 11190 (PR 236 ms), 17072 (deviated axes).")

### Block M6.7: Freeze + non-regression check

In [ ]:
# Freeze M6 (combined notebook) under standardized artifact names (do NOT clobber the legacy
# 05 outputs). Non-regression check against the frozen reference numbers.
import os, json, joblib
from datetime import datetime

final_raw = make_xgb(42).fit(X, y)
final_cal = CalibratedClassifierCV(make_xgb(42), method='sigmoid', cv=3).fit(X, y)
joblib.dump(final_raw, os.path.join(MODELS, 'm6_xgb_raw.joblib'))
joblib.dump(final_cal, os.path.join(MODELS, 'm6_calibrated.joblib'))

config = {
    'model': 'M6_marquette',
    'role': 'External reference (Marquette 12SL) — NOT in the ensemble, never touches fold 10. PTB-XL only.',
    'representation': 'C_combined (6 clinical from A + 4 discovery from B)',
    'algo': 'XGBoost max_depth=2, lr=0.05, n_estimators=100, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, min_child_weight=3',
    'features': feats,
    'calibration': 'Platt (sigmoid), cv=3 internal',
    'eval_protocol': 'OOF on native PTB-XL folds 1-8 (no shuffle); standardized fold-9 eval added for comparability',
    'headline_oof_folds1_8': {'AUC': round(AUC_oof,4), 'AUC_IC95': AUC_IC, 'AP': round(AP_oof,4), 'AP_IC95': AP_IC, 'threshold_F1max': THR},
    'multiseed_oof': MULTISEED,
    'metrics_standard_fold9': M6_STD,
    'fold10': 'NEVER touched — reserved for final ensemble evaluation',
    'date_frozen': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'files': {'raw': 'm6_xgb_raw.joblib', 'calibrated': 'm6_calibrated.joblib',
              'metrics_fold9': 'reports/metrics/M6_marquette_metrics.json'},
}
json.dump(config, open(os.path.join(MODELS, 'm6_marquette_config.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

# Non-regression vs the frozen reference (AUC 0.9686 / AP 0.5834 / F1 0.679 on OOF folds 1-8)
assert abs(AUC_oof - 0.9686) < 0.010, f"AUC regression: {AUC_oof:.4f}"
assert abs(AP_oof  - 0.5834) < 0.030, f"AP regression: {AP_oof:.4f}"
assert abs(best.f1 - 0.679)  < 0.030, f"F1 regression: {best.f1:.4f}"
print(f"[OK] M6 frozen -> m6_marquette_config.json | non-regression PASSED "
      f"(AUC {AUC_oof:.4f}, AP {AP_oof:.4f}, F1 {best.f1:.3f})")